# Cleaning the job ads

The collection in `01_collect.ipynb` produced 2,119 ads with full texts. Before
skills can be extracted, the data needs work: some ads are duplicates, some
fields contain nested structures, and some ads may not be data roles at all.

This notebook goes through these problems one at a time and saves a clean table
to `data/processed/`. Like the raw data, it is not published on GitHub.

## Setup

The full texts come from `details.jsonl`. The search terms that found each ad
are only stored in `search_results.json`, but they are needed later to compare
roles, so both files are loaded.

In [ ]:
import json
from pathlib import Path

import pandas as pd

RAW_DIR = Path.cwd().parent / "data" / "raw"
PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

details = pd.read_json(RAW_DIR / "details.jsonl", lines=True)
search = json.loads((RAW_DIR / "search_results.json").read_text(encoding="utf-8"))

print(details.shape)
print(len(search), "ads in the search results")

## Overview: dtypes and missing values

All 42 fields have been loaded into the `details` DataFrame. Now we can inspect what data types pandas has inferred for each, and how often values are missing.


In [ ]:
overview = pd.DataFrame({
    "dtypes": details.dtypes,
    "missing": details.isna().sum(),
    "missing_%": (details.isna().mean() * 100).round(1)
})

overview.sort_values("missing", ascending=False)

**The fields the analysis needs are complete.** The description, the title, the
main occupation (`hauptberuf`), the locations and the publication date are
filled in every ad. The employer (`firma`) is missing in only one ad.

**27 of the 42 fields have gaps, and some are almost empty:**

- **Salary.** Only 257 ads (12%) give a salary range (`gehaltsspanneVon`/`Bis`),
  and only 31 a fixed salary (`festgehalt`). Salary can therefore not be
  analysed reliably and is not one of the project's questions.
- **Contract details.** `befristetBis`, `befristungInMonaten` and
  `homeofficeprozent` are missing in over 94% of the ads. The complete field
  `vertragsdauer` covers the contract type instead.
- **Technical fields.** `chiffrenummer`, `externeURL`, `arbeitgeberKundennummerHash`
  and the `allianzpartner` fields describe how the ad was published, not the job.

**The yes/no fields have a different kind of gap.** `homeofficemoeglich` is
empty in 45% of the ads, the part-time fields in about a third. An empty field
probably means "not specified" rather than "no", so it cannot simply be filled
with 0. The next cell checks what values these fields actually contain.

**Two date fields are stored as text** (`datumErsteVeroeffentlichung`,
`aenderungsdatum`) and need converting to dates.

In [ ]:
for col in ["verguetungsangabe", "vertragsdauer", "stellenangebotsart",
            "homeofficemoeglich", "arbeitszeitVollzeit"]:
    print(details[col].value_counts(dropna=False), "\n")

**`stellenangebotsart` has only one value** (`ARBEIT`, regular employment) — the
filter `angebotsart = 1` in the collection worked. The field carries no
information and is dropped.

**`vertragsdauer` is usable, with its own "no information" category:** 58% of
the ads are permanent, 7% fixed-term, and 35% say nothing. Salary works the same
way: `verguetungsangabe` shows that 1,831 ads (86%) give no salary information.

**The yes/no fields have three states, not two.** `homeofficemoeglich` contains
`1.0` (359 ads), `0.0` (807 ads) and empty (953 ads). Since an explicit "no"
exists, an empty field means "not specified", not "no". This matters for the
numbers: home office is offered in 17% of all ads, but in 31% of the ads that
say anything about it. The empty fields are therefore kept as missing, not filled with 0.

## Selecting the columns for analysis

The project's questions are about skills, roles and seniority, so the analysis
needs the text, the title, the occupation and a few descriptive fields. Fields
that are almost empty, constant or purely technical are dropped.

`istPrivateArbeitsvermittlung` (posted by a recruitment agency) is kept for now:
the duplicates seen during the collection came from an agency, and this field
can show whether that is a pattern.

The search terms that found each ad are added from the search results, so that
roles can be compared later.

In [ ]:
keep = [
    "referenznummer", "stellenangebotsTitel", "firma", "hauptberuf",
    "stellenangebotsBeschreibung", "stellenlokationen",
    "datumErsteVeroeffentlichung", "vertragsdauer",
    "arbeitszeitVollzeit", "homeofficemoeglich", "istPrivateArbeitsvermittlung",
]

jobs = details[keep].copy()

# add the search terms from the search results
terms = pd.DataFrame(search)[["referenznummer", "suchbegriffe"]]
jobs = jobs.merge(terms, on="referenznummer", how="left")

print(jobs.shape)
print(jobs["suchbegriffe"].isna().sum(), "ads without search terms")
jobs.head()

The table now has 12 columns: the 11 selected fields plus `suchbegriffe`. Every
ad found its search terms, so the two files match completely.

The first rows show the Berlin pair again (rows 3 and 4): same agency, same
title, same text, different postcodes — and also different publication dates
(14 September and 4 August). Both are flagged as posted by a private recruitment
agency (`istPrivateArbeitsvermittlung = 1.0`). This is examined in the
duplicates section.

## Fixing the boolean and date types

The yes/no fields are converted to pandas' `boolean` type, which can hold
True, False *and* a missing value, so "not specified" is kept. The publication
date is converted from text to a date.

In [ ]:
# convert yes/no fields to boolean
for col in ["arbeitszeitVollzeit", "homeofficemoeglich", "istPrivateArbeitsvermittlung"]:
    jobs[col] = jobs[col].astype("boolean")
    
# convert the publication date to datetime
jobs["datumErsteVeroeffentlichung"] = pd.to_datetime(jobs["datumErsteVeroeffentlichung"])

print(jobs.dtypes, "\n")
print(jobs["homeofficemoeglich"].value_counts(dropna=False), "\n")
print(jobs["datumErsteVeroeffentlichung"].min(), "to", jobs["datumErsteVeroeffentlichung"].max())

The conversion lost nothing: the home office counts are the same as before
(359 True, 807 False, 953 missing), and missing values are now shown as `<NA>`.

**The publication dates is a surprise, though.** The newest ad is from 7 October 2026,
the day of the collection. The oldest was first published on 31 March 2021 —
five and a half years earlier, and still online. Such ads may be positions that
stay open for a long time, or "evergreen" postings that are kept online to
collect applications rather than to fill one specific job. The next cell shows
how many ads are this old.

In [ ]:
collected = pd.Timestamp("2026-10-07")
age_days = (collected - jobs["datumErsteVeroeffentlichung"]).dt.days

print(age_days.describe(), "\n")

bins = [-1, 30, 90, 180, 365, 10_000]
labels = ["up to 1 month", "1-3 months", "3-6 months", "6-12 months", "over 1 year"]
print(pd.cut(age_days, bins=bins, labels=labels).value_counts(sort=False))

**Most ads are recent.** Half were first published in the last 34 days (the
median), and 77% in the last three months (1,028 + 601 of 2,119 ads). The mean
of 77 days is more than twice the median: a few very old ads pull it up. The
distribution is strongly skewed to the right.

**81 ads (3.8%) are older than a year**, the oldest over five years (2,016 days).

**Decision: all ads are kept.** Every one of them was online on the day of the
collection, so all of them are part of what employers currently advertise. Any
cutoff (six months? a year?) would be arbitrary. Instead, the age of each ad is
stored as a column, so the analysis can later check whether the old ads change
the results.

In [ ]:
# Insert the age of each ad as a new column.
jobs["age_days"] = age_days

jobs["age_days"].describe()

## Unpacking the locations

`stellenlokationen` holds a list of locations per ad, each a nested dictionary.
Before extracting the city and federal state, a look at its structure: how many
locations do ads have, and what does one entry contain?

In [ ]:
# Number of locations per ad
n_locations = jobs["stellenlokationen"].apply(len)
print(n_locations.value_counts().sort_index(), "\n")

jobs["stellenlokationen"].iloc[0]

**Most ads have a single location (1,897, or 90%), but 222 list several** — up
to ten. The count at exactly ten (22 ads) is higher than at nine (11 ads), which
suggests the job search may cut longer lists off at ten. If so, some ads are
advertised in even more places than the data shows.

Each location contains an address (street, house number, postcode, city,
federal state, country) and coordinates. Not every entry has every part: the
first ad has a street, others start with the postcode.

**How multi-location ads are handled:** one ad is one job posting, so for
counting skills it counts once, no matter how many locations it lists. For the
location columns, the first location is used, and the number of locations is
stored separately.

The address fields are taken from the first location. `.get()` returns `None`
instead of an error when a field is missing.

In [ ]:
def first_location(locations, key):
    """Return one address field of the first location, or None if it is missing."""
    if not locations:
        return None
    return locations[0].get("adresse", {}).get(key)

jobs["n_locations"] = n_locations
for key, name in [("plz", "postcode"), ("ort", "city"), ("region", "state"), ("land", "country")]:
    jobs[name] = jobs["stellenlokationen"].apply(first_location, key=key)

print(jobs[["postcode", "city", "state", "country"]].isna().sum(), "\n")
print(jobs["country"].value_counts(), "\n")
print(jobs["state"].value_counts())

**Not all ads are in Germany.** Based on the first location, 2,031 ads (95.8%)
are in Germany, but 88 are abroad: 81 in Austria, 4 in Switzerland and one
each in Slovakia, Israel and Spain. The Bundesagentur's job search also
publishes ads from other countries.

All 16 German federal states appear, led by Bayern (429), Nordrhein-Westfalen
(405) and Baden-Württemberg (299). A few ads have no postcode (178), city (69)
or federal state (64); one has the state `SONSTIGES` (other).

**Decision: the analysis covers Germany only**, because the project asks what
*German* employers expect. One detail has to be checked first: the country
above comes from the first location only. An ad with several locations might
list a foreign location first and a German one later. Such an ad belongs in
the dataset.

In [ ]:
def has_german_location(locations):
    """True if at least one of the ad's locations is in Germany."""
    return any(loc.get("adresse", {}).get("land") == "DEUTSCHLAND" for loc in locations)

in_germany = jobs["stellenlokationen"].apply(has_german_location)
print(in_germany.value_counts(), "\n")

# ads that list Germany, but not as their first location
print(((jobs["country"] != "DEUTSCHLAND") & in_germany).sum(), "ads list Germany, but not first")

The two counts agree: 2,031 ads have a German location, and no ad lists Germany
only as a later location. So the first location is a reliable basis for the
location columns, and the filter can simply keep the ads in Germany.

In [ ]:
# Keep only ads with a German location and drop the country column
jobs = jobs[in_germany].drop(columns="country").reset_index(drop=True)

print(jobs.shape)
print(jobs[["postcode", "city", "state"]].isna().sum())

## Duplicates

The collection found exact duplicate descriptions: ads with different
reference numbers but identical text. Before deciding what to do with them, a
look at what these groups have in common: the same employer? the same title?
different locations?

In [ ]:
desc = jobs["stellenangebotsBeschreibung"]
dup = jobs[desc.duplicated(keep=False)]

print(desc.duplicated().sum(), "duplicate descriptions")
print(len(dup), "ads involved, sharing", dup["stellenangebotsBeschreibung"].nunique(), "different texts\n")

groups = (
    dup.groupby("stellenangebotsBeschreibung")
    .agg(
        n_ads=("referenznummer", "size"),
        employer=("firma", "first"),
        title=("stellenangebotsTitel", "first"),
        n_titles=("stellenangebotsTitel", "nunique"),
        n_cities=("city", "nunique"),
        n_postcodes=("postcode", "nunique"),
        agency=("istPrivateArbeitsvermittlung", "first"),
    )
    .sort_values("n_ads", ascending=False)
    .reset_index(drop=True)
)
groups

**After the Germany filter, 32 descriptions are duplicates** (1.6% of 2,031
ads). They form 25 groups of identical texts: 18 pairs and 7 groups of three,
57 ads in total. Within every group the title is identical too.

**Two kinds of duplicates:**

- **The same job in several places (7 groups).** The ads differ only in their
  location, e.g. Staffbase and B&O Service with three cities each, or the
  Berlin pair from the collection with two postcodes. This is one job
  advertised for several locations.
- **The same job posted again (16 groups).** Same text, same title, same
  location — the ad was simply published more than once. For 2 more groups the
  location is missing, so the two cases cannot be told apart.

**One employer stands out: Akkodis**, an engineering service provider, accounts
for 10 of the 25 groups.

**Decision:** for counting skills, each text counts once, regardless of how many
copies or locations it has. Otherwise a job posted three times would count three
times. Before removing the copies, their number is stored in `n_copies`, so
this information is not lost.

**A finding for the next step:** many titles here are not data roles at all —
*Mechanical Design Engineer*, *Power Electronics Engineer*, *Senior Seat
Integration Engineer*, *Head of HR IT*. This confirms the suspicion from the
collection: broad search terms such as "Data Engineer" also bring in other
engineering and IT positions.

In [ ]:
jobs["n_copies"] = desc.map(desc.value_counts())
jobs = jobs.drop_duplicates(subset="stellenangebotsBeschreibung", keep="first").reset_index(drop=True)

print(jobs.shape)
print(jobs["n_copies"].value_counts().sort_index())

The numbers match: 2,031 − 32 = 1,999 ads remain, and `n_copies` records 18 ads
that had one copy and 7 that had two.

The check above only catches *identical* texts. If an employer changes a single
word, the copy is missed. A second, looser check: ads that share employer and
title but have different texts.

In [ ]:
same_job = jobs.duplicated(subset=["firma", "stellenangebotsTitel"], keep=False)
print(same_job.sum(), "ads share employer and title with another ad\n")

(jobs[same_job]
 .groupby(["firma", "stellenangebotsTitel"])
 .size()
 .sort_values(ascending=False)
 .head(15))

**97 ads share employer and title with another ad, but not their text.** Most
come from a few employers: CHECK24 has two trainee programmes with 7 ads each,
and KUMAVISION has six job titles with 3 to 5 ads each. Whether these are the
same job with small text changes (for example the city named in the text) or
really different positions needs a look at the texts themselves. This is
checked in the next part.